<a href="https://colab.research.google.com/github/gd-Sahat/ClockBiasPINN/blob/main/model_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd

clkbias_diff = pd.read_csv('/content/drive/MyDrive/CLOCKBIASPINN/clkbias_diff.gz', compression='gzip')
clkbias_diff.head()

,id,timestamp,bias_interp_s,bias_s,bias_diff,bias_interp_ns,bias_ns,bias_diff_ns
0,G01,2023-05-01 00:00:00,0.000189,0.000189,-2.555663e-09,188869.424164,188866.868501,-2.555663
1,G01,2023-05-01 00:00:30,0.000189,0.000189,-2.543851e-09,188869.332078,188866.788226,-2.543851
2,G01,2023-05-01 00:01:00,0.000189,0.000189,-2.534747e-09,188869.239991,188866.705245,-2.534747
3,G01,2023-05-01 00:01:30,0.000189,0.000189,-2.547661e-09,188869.147905,188866.600244,-2.547661
4,G01,2023-05-01 00:02:00,0.000189,0.000189,-2.551347e-09,188869.055819,188866.504471,-2.551347


In [5]:
!pip install tensorflow --quiet

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


In [6]:
df = clkbias_diff.copy()

# 3.0 Ensure timestamp is datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 3.1 Encode satellite ID as integers
df['id_enc'] = pd.factorize(df['id'])[0]

# 3.2 Normalize bias_interp_ns
from sklearn.preprocessing import StandardScaler

scaler_x = StandardScaler()
df['bias_interp_ns_norm'] = scaler_x.fit_transform(df[['bias_interp_ns']])

# 3.3 Turn timestamp into a continuous feature (seconds since start)
t0 = df['timestamp'].min()
df['time_s'] = (df['timestamp'] - t0).dt.total_seconds().astype(np.float32)

# ── NEW: standardize time_s ───────────────────────────────────
scaler_t = StandardScaler()
df['time_s_norm'] = scaler_t.fit_transform(df[['time_s']]).astype(np.float32)
# (optional) drop the raw seconds column to save memory
df.drop(columns='time_s', inplace=True)
# ───────────────────────────────────────────────────────────────

# 3.4 Select features & raw target
features = df[['id_enc', 'time_s_norm', 'bias_interp_ns_norm']].values.astype(np.float32)
targets  = df['bias_diff_ns'].values.astype(np.float32)

# ── 3.5 SCALE THE TARGET HERE ───────────────────────────────────
y_scaler       = StandardScaler()
targets_scaled = y_scaler.fit_transform(targets.reshape(-1,1)).ravel().astype(np.float32)
print("target mean/std:", targets_scaled.mean(), targets_scaled.std())
# ────────────────────────────────────────────────────────────────

target mean/std: 2.3353003e-08 0.9999992


In [7]:
df

,id,timestamp,bias_interp_s,bias_s,bias_diff,bias_interp_ns,bias_ns,bias_diff_ns,id_enc,bias_interp_ns_norm,time_s_norm
0,G01,2023-05-01 00:00:00,0.000189,0.000189,-2.555663e-09,188869.424164,188866.868501,-2.555663,0,0.474967,-1.721237
1,G01,2023-05-01 00:00:30,0.000189,0.000189,-2.543851e-09,188869.332078,188866.788226,-2.543851,0,0.474967,-1.721235
2,G01,2023-05-01 00:01:00,0.000189,0.000189,-2.534747e-09,188869.239991,188866.705245,-2.534747,0,0.474967,-1.721233
3,G01,2023-05-01 00:01:30,0.000189,0.000189,-2.547661e-09,188869.147905,188866.600244,-2.547661,0,0.474967,-1.721232
4,G01,2023-05-01 00:02:00,0.000189,0.000189,-2.551347e-09,188869.055819,188866.504471,-2.551347,0,0.474966,-1.721230
...,...,...,...,...,...,...,...,...,...,...,...
66029711,G32,2025-05-01 23:57:30,-0.000463,-0.000463,-1.760195e-09,-463035.542680,-463037.302875,-1.760195,31,-1.155556,1.735882
66029712,G32,2025-05-01 23:58:00,-0.000463,-0.000463,-1.774284e-09,-463035.245957,-463037.020241,-1.774284,31,-1.155555,1.735884
66029713,G32,2025-05-01 23:58:30,-0.000463,-0.000463,-1.778575e-09,-463034.949235,-463036.727809,-1.778575,31,-1.155555,1.735886
66029714,G32,2025-05-01 23:59:00,-0.000463,-0.000463,-1.771546e-09,-463034.652512,-463036.424058,-1.771546,31,-1.155554,1.735887


In [8]:
# Colab cell 4 (memory-efficient sliding windows)
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

SEQ_LEN = 32

from numpy.lib.stride_tricks import sliding_window_view

all_windows = sliding_window_view(features, window_shape=SEQ_LEN, axis=0)[:-1]
X = all_windows.transpose(0,2,1)           # (N-SEQ_LEN, 32, 3)
y = targets_scaled[SEQ_LEN:]               # (N-SEQ_LEN,)

# train/val split
split   = int(0.8 * len(X))
X_train = X[:split]
y_train = y[:split]
X_val   = X[split:]
y_val   = y[split:]


In [9]:
# At the top of your notebook, once per session:
from google.colab import drive
drive.mount('/content/drive')

# …after Cell 3/4, once you have X_train, y_train, X_val, y_val:
import numpy as np

np.save('/content/drive/MyDrive/CLOCKBIASPINN/X_train.npy', X_train)
np.save('/content/drive/MyDrive/CLOCKBIASPINN/y_train.npy', y_train)
np.save('/content/drive/MyDrive/CLOCKBIASPINN/X_val.npy',   X_val)
np.save('/content/drive/MyDrive/CLOCKBIASPINN/y_val.npy',   y_val)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
# Colab cell 5 (fixed TransformerBlock)
class PositionalEmbedding(layers.Layer):
    def __init__(self, seq_len, d_model):
        super().__init__()
        self.d_model = d_model
        self.proj = layers.Dense(d_model)
        self.pos_emb = layers.Embedding(input_dim=seq_len, output_dim=d_model)
        self.seq_len = seq_len

    def call(self, x):
        x = self.proj(x)                                     # (B, T, d_model)
        positions = tf.range(self.seq_len)[tf.newaxis, :]    # (1, T)
        pos_emb = self.pos_emb(positions)                    # (1, T, d_model)
        return x + pos_emb

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads, key_dim=d_model)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(d_model),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(rate)
        self.drop2 = layers.Dropout(rate)

    def call(self, inputs, training=None):
        # 1) Self‐attention + skip + norm
        attn_output = self.att(inputs, inputs)
        attn_output = self.drop1(attn_output, training=training)
        out1 = self.norm1(inputs + attn_output)

        # 2) Feed‐forward + skip + norm
        ffn_output = self.ffn(out1)
        ffn_output = self.drop2(ffn_output, training=training)
        return self.norm2(out1 + ffn_output)


In [1]:
# Colab cell 6 (modified)
import tensorflow as tf
from tensorflow.keras import Input, Model, layers, optimizers

# Hyper-parameters
d_model   = 64
num_heads = 4
ff_dim    = 128

# Inferred from your data
SEQ_LEN     = X_train.shape[1]   # 32
FEATURE_DIM = X_train.shape[2]   # 3

# 1) Input + positional embed
inp = Input(shape=(SEQ_LEN, FEATURE_DIM))
x   = PositionalEmbedding(SEQ_LEN, d_model)(inp)

# 2) Two stacked Transformer blocks
x   = TransformerBlock(d_model, num_heads, ff_dim)(x)
x   = TransformerBlock(d_model, num_heads, ff_dim)(x)

# 3) Pool & head
x   = layers.GlobalAveragePooling1D()(x)
x   = layers.Dropout(0.1)(x)
x   = layers.Dense(64, activation="relu")(x)
x   = layers.Dropout(0.1)(x)
out = layers.Dense(1)(x)

# 4) Assemble & compile with lower LR + gradient clipping
model = Model(inp, out)
opt   = optimizers.Adam(learning_rate=1e-4, clipnorm=1.0)
model.compile(optimizer=opt, loss="mse", metrics=["mae"])

model.summary()


NameError: name 'X_train' is not defined

In [ ]:
BATCH_SIZE = 32
AUTOTUNE    = tf.data.AUTOTUNE

train_ds = (
  tf.data.Dataset
    .from_tensor_slices((X_train, y_train))
    .shuffle(50_000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
  tf.data.Dataset
    .from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

model.fit(
  train_ds,
  epochs=20,
  validation_data=val_ds
)
